## Inspeccionar datos

In [11]:
!pwd
import pandas as pd

/home/daniel-linux/Tesis/Recolectar_Datos/notebooks


In [12]:
import pandas as pd
import glob
import os
import numpy as np
from tqdm import tqdm  # ¡Agregado para barras de progreso!

# Ruta a los archivos parquet
parquet_pattern = "../data/intermediate/parquet/*.parquet"

# Buscar todos los archivos parquet que coincidan con el patrón
parquet_files = glob.glob(parquet_pattern)

# Inicializar contadores globales para offsets
global_entity_counter = 1
global_unique_entity_counter = 1
global_sentence_counter = 1

# Lista para almacenar los DataFrames procesados
processed_dfs = []

KeyboardInterrupt: 

In [ ]:
# Procesar archivos con barra de progreso
for file in tqdm(parquet_files, desc="Procesando archivos"):
    # Cargar el DataFrame del archivo actual
    df = pd.read_parquet(file)
    
    # Para sentence_id: mapear valores únicos locales a globales con offset
    local_sentences = sorted(df['sentence_id'].unique())
    sentence_map = {local_id: f"SENT_{str(global_sentence_counter + i).zfill(5)}" 
                    for i, local_id in enumerate(local_sentences)}
    df['global_sentence_id'] = df['sentence_id'].map(sentence_map)
    # Actualizar contador global
    global_sentence_counter += len(local_sentences)
    
    # Para unique_entity_id: mapear valores únicos locales a globales con offset
    local_unique_entities = sorted(df['unique_entity_id'].unique())
    unique_entity_map = {local_id: f"UENT_{str(global_unique_entity_counter + i).zfill(5)}" 
                         for i, local_id in enumerate(local_unique_entities)}
    df['global_unique_entity_id'] = df['unique_entity_id'].map(unique_entity_map)
    # Actualizar contador global
    global_unique_entity_counter += len(local_unique_entities)
    
    # Para entity_id: como es único por fila en el archivo, asignar globales secuenciales para estas filas
    n_rows = len(df)
    df['global_entity_id'] = [f"ENT_{str(global_entity_counter + i).zfill(5)}" 
                              for i in range(n_rows)]
    # Actualizar contador global
    global_entity_counter += n_rows
    
    # Agregar el DF procesado a la lista
    processed_dfs.append(df)

Procesando archivos:   0%|          | 0/27 [00:00<?, ?it/s]

Procesando archivos:  26%|██▌       | 7/27 [01:06<03:09,  9.48s/it]


KeyboardInterrupt: 

In [ ]:
# Unificar todos los DataFrames procesados
unified_df = pd.concat(processed_dfs, ignore_index=True)

# Opcional: eliminar columnas originales si no las necesitas
# unified_df = unified_df.drop(columns=['entity_id', 'unique_entity_id', 'sentence_id'])

# Mostrar las primeras filas para verificar
print(unified_df[['global_entity_id', 'global_unique_entity_id', 'global_sentence_id', 'entity', 'type', 'start', 'end', 'sentence', 'iob_tag', 'token_start', 'token_end', 'tokens']].head(10))

# Configuración para guardar en partes
output_dir = "../data/unified/"
os.makedirs(output_dir, exist_ok=True)
chunk_size = 250000  # Ajusta si es necesario
num_chunks = (len(unified_df) + chunk_size - 1) // chunk_size

In [ ]:
# Guardar en chunks con barra de progreso
for i in tqdm(range(num_chunks), desc="Guardando chunks"):
    start_idx = i * chunk_size
    end_idx = min((i + 1) * chunk_size, len(unified_df))
    chunk_df = unified_df.iloc[start_idx:end_idx]
    
    # Nombre del archivo: unified_part_{i+1}.parquet
    chunk_file = os.path.join(output_dir, f"unified_part_{i+1:03d}.parquet")
    chunk_df.to_parquet(chunk_file, index=False)
    print(f"Guardado chunk {i+1}/{num_chunks} en {chunk_file} ({len(chunk_df)} filas)")

print(f"¡Unificación completada! Total de filas: {len(unified_df)}. Archivos guardados en {output_dir}")

In [ ]:
df

,entity_id,unique_entity_id,sentence_id,entity,type,start,end,sentence,iob_tag,token_start,token_end,tokens
0,ENT_00001,UENT_00001,SENT_00001,5,QUANTITY,0,1,5 ounces rum 4 ounces triple sec 3 ounces Tia ...,B-QUANTITY,0,0,"[5, ounces, rum, 4, ounces, triple, sec, 3, ou..."
1,ENT_00002,UENT_00002,SENT_00001,ounces,UNIT,2,8,5 ounces rum 4 ounces triple sec 3 ounces Tia ...,B-UNIT,1,1,"[5, ounces, rum, 4, ounces, triple, sec, 3, ou..."
2,ENT_00003,UENT_00003,SENT_00001,rum,FOOD,9,12,5 ounces rum 4 ounces triple sec 3 ounces Tia ...,B-FOOD,2,2,"[5, ounces, rum, 4, ounces, triple, sec, 3, ou..."
3,ENT_00004,UENT_00004,SENT_00001,4,QUANTITY,13,14,5 ounces rum 4 ounces triple sec 3 ounces Tia ...,B-QUANTITY,3,3,"[5, ounces, rum, 4, ounces, triple, sec, 3, ou..."
4,ENT_00005,UENT_00002,SENT_00001,ounces,UNIT,15,21,5 ounces rum 4 ounces triple sec 3 ounces Tia ...,B-UNIT,4,4,"[5, ounces, rum, 4, ounces, triple, sec, 3, ou..."
...,...,...,...,...,...,...,...,...,...,...,...,...
13357,ENT_13358,UENT_00761,SENT_00700,Parmesan cheese,FOOD,134,149,14 ounces artichoke hearts (drained and choppe...,B-FOOD I-FOOD,21,22,"[14, ounces, artichoke, hearts, (drained, and,..."
13358,ENT_13359,UENT_00892,SENT_00700,2 1/2,QUANTITY,150,155,14 ounces artichoke hearts (drained and choppe...,B-QUANTITY I-QUANTITY,23,24,"[14, ounces, artichoke, hearts, (drained, and,..."
13359,ENT_13360,UENT_00093,SENT_00700,cups,UNIT,156,160,14 ounces artichoke hearts (drained and choppe...,B-UNIT,25,25,"[14, ounces, artichoke, hearts, (drained, and,..."
13360,ENT_13361,UENT_00058,SENT_00700,shredded,PROCESS,161,169,14 ounces artichoke hearts (drained and choppe...,B-PROCESS,26,26,"[14, ounces, artichoke, hearts, (drained, and,..."


## Datos intermedios

In [ ]:
import os 
os.chdir('..')
!pwd

In [ ]:
# Celda 1: Importar librerías y cargar el archivo
import pandas as pd
import numpy as np

# Cargar el archivo mergeado
df_merged = pd.read_csv('./data/intermediate/merged_recipes_three_columns.csv', sep = ';')
print("✅ Archivo cargado exitosamente")
print(f"📊 Dimensiones del DataFrame: {df_merged.shape}")
print(f"📋 Columnas: {list(df_merged.columns)}")

In [ ]:
# Mostrar más filas y columnas para ver el DataFrame más grande
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 200)

In [ ]:
# Celda 2: Mostrar información básica
print("📝 Información del DataFrame:")
df_merged.info()

print("\n🧮 Valores nulos por columna:")
print(df_merged.isnull().sum())

In [ ]:
# Celda 3: Mostrar primeras y últimas filas
print("🔍 Primeras 5 filas:")
display(df_merged.head())

print("\n🔍 Últimas 5 filas:")
display(df_merged.tail())

In [ ]:
# Celda 4: Estadísticas de la unión
if '_merge' in df_merged.columns:
    print("📈 Estadísticas de la unión:")
    merge_stats = df_merged['_merge'].value_counts()
    print(merge_stats)
    
    print("\n📊 Porcentajes:")
    for key, value in merge_stats.items():
        print(f"{key}: {value/len(df_merged)*100:.2f}%")
else:
    print("ℹ️ No se encontró columna '_merge'")

In [ ]:
# Celda 5: Ver ejemplos de cada tipo de registro
print("\n🎯 Ejemplos de registros:")

# Si existe columna _merge
if '_merge' in df_merged.columns:
    print("\n📍 Coincidencias exactas (both):")
    both_examples = df_merged[df_merged['_merge'] == 'both'].head(2)
    display(both_examples)
    
    print("\n📍 Solo en RecipeNLG (left_only):")
    left_examples = df_merged[df_merged['_merge'] == 'left_only'].head(2)
    display(left_examples)
    
    print("\n📍 Solo en 3A2M (right_only):")
    right_examples = df_merged[df_merged['_merge'] == 'right_only'].head(2)
    display(right_examples)

In [ ]:
# Celda 6: Verificar duplicados en las columnas clave
print("\n🔎 Duplicados en columnas clave:")
key_columns = ['title', 'ingredients', 'NER']
for col in key_columns:
    if col in df_merged.columns:
        duplicates = df_merged[col].duplicated().sum()
        print(f"{col}: {duplicates} duplicados ({duplicates/len(df_merged)*100:.1f}%)")

In [ ]:
# Celda 7: Estadísticas descriptivas básicas
print("\n📊 Estadísticas descriptivas:")
print("Columnas numéricas:")
numeric_cols = df_merged.select_dtypes(include=[np.number]).columns
if len(numeric_cols) > 0:
    display(df_merged[numeric_cols].describe())
else:
    print("No hay columnas numéricas")

In [ ]:
# Celda 8: Guardar un resumen rápido
print("💾 Guardando resumen rápido...")
summary_path = './data/merged_file_summary.txt'
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write(f"Resumen del archivo mergeado\n")
    f.write(f"============================\n")
    f.write(f"Fecha: {pd.Timestamp.now()}\n")
    f.write(f"Filas: {df_merged.shape[0]}\n")
    f.write(f"Columnas: {df_merged.shape[1]}\n")
    f.write(f"Columnas: {list(df_merged.columns)}\n\n")
    
    f.write("Valores nulos por columna:\n")
    for col, null_count in df_merged.isnull().sum().items():
        f.write(f"{col}: {null_count}\n")
    
    if '_merge' in df_merged.columns:
        f.write(f"\nEstadísticas de unión:\n")
        for key, value in df_merged['_merge'].value_counts().items():
            f.write(f"{key}: {value} ({value/len(df_merged)*100:.2f}%)\n")

print(f"✅ Resumen guardado en: {summary_path}")

In [ ]:
# Celda 1: Importar librerías y cargar el archivo
import pandas as pd
import numpy as np

# Cargar el archivo mergeado
df_merged = pd.read_csv('./data/intermediate/tasteset_entities.csv',sep=';')
print("✅ Archivo cargado exitosamente")
print(f"📊 Dimensiones del DataFrame: {df_merged.shape}")
print(f"📋 Columnas: {list(df_merged.columns)}")


In [ ]:
df_merged

In [ ]:
!pwd

In [ ]:
# Celda 1: Importar librerías y cargar el archivo
import pandas as pd
import numpy as np

# Cargar el archivo mergeado
df_merged = pd.read_csv('./data/intermediate/humble_intelligence_1million_entities.csv',sep=';')
print("✅ Archivo cargado exitosamente")
print(f"📊 Dimensiones del DataFrame: {df_merged.shape}")
print(f"📋 Columnas: {list(df_merged.columns)}")

In [ ]:
df_merged